# Preparing the war-photo-recall data

This is a presentation copy of the complete research preparation procedure, including its experiment-specific scoring reconciliation. The private source workbooks are not included. Saved aggregate checks below come from the original notebook's execution; this copy has not been rerun against those workbooks.

Source: `githubpsyche/emcr3`, `data/Dupertuys2026_data_preparation.ipynb`, revision `c963a8158f0dfca436d8b304328759c59766223d` ([original, repository access required](https://github.com/githubpsyche/emcr3/blob/c963a8158f0dfca436d8b304328759c59766223d/data/Dupertuys2026_data_preparation.ipynb)). Changes are confined to paths/configuration, cell organization and public presentation; the scoring calculations and assertions are retained.

After installing JAXCMR, this example also requires `pandas` and `openpyxl` for the existing spreadsheet reader (`python -m pip install pandas openpyxl`). Set `source_dir` to your local copy of the experiment's source folder. Relative paths use `project_root`, the enclosing Git root, or the kernel's working folder; on Windows use forward slashes or raw strings for absolute paths.

In [ ]:
#| code-summary: "1 — Configure source and output folders"

project_root = None
source_dir = "private/war-photo-recall"
output_dir = "results/tutorial/war-photo-preparation"
MAX_LIST_LENGTH = 12


In [ ]:
#| code-summary: "2 — Install packages when running in Colab"

import sys

if "google.colab" in sys.modules:
    import subprocess

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "pandas", "openpyxl",
        "jaxcmr @ git+https://github.com/githubpsyche/jaxcmr.git@main",
    ])


In [ ]:
#| code-summary: "3 — Import packages"

import hashlib
import json
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import h5py
from jax import jit
import jax.numpy as jnp
import numpy as np
import pandas as pd

from jaxcmr.analyses import crp
from jaxcmr.helpers import (
    apply_by_subject,
    find_project_root,
    load_data,
    resolve_project_path,
    save_dict_to_hdf5,
)


In [ ]:
#| code-summary: "4 — Locate the private source files and separate output folder"

if project_root is not None:
    root_path = resolve_project_path(project_root, Path.cwd())
else:
    try:
        root_path = Path(find_project_root())
    except FileNotFoundError:
        root_path = Path.cwd()
output_path = resolve_project_path(output_dir, root_path)

LAB_ROOT = resolve_project_path(source_dir, root_path)
if not LAB_ROOT.is_dir():
    raise FileNotFoundError("Set source_dir to the separately held lab source folder.")
PROJECT_ROOT = root_path
OUTPUT_DIR = output_path
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MEMORY_ROOT = LAB_ROOT / 'Memory experiment'
FINAL_WORKBOOK = MEMORY_ROOT / 'Data Analysis' / 'Final data files' / 'Completed_Recall_Data2024.xlsx'
SCORING_MEMORY = MEMORY_ROOT / 'SCORING_MEMORY'
SIM_EXP1 = LAB_ROOT / 'Similarity Exp 1 - UK Sample' / "Juliette's stuff"
SCORING_JULIETTE = SCORING_MEMORY / 'rec_itemnos_juliette.xlsx'
SCORING_DUSAN = SCORING_MEMORY / 'rec_itemnos_dusan.xlsx'
SCORING_DISCREPANCIES = SCORING_MEMORY / 'scoring_discrepancies.xlsx'
SCORING_DISCREPANCIES_FILTERED = SCORING_MEMORY / 'scoring_discrepancies_filtered_FINAL.xlsx'


In [ ]:
#| code-summary: "5 — Describe the original workbook columns and codes"

PRES_COLS = [f'Pres{i}' for i in range(1, MAX_LIST_LENGTH + 1)]
OP_COLS = [f'OP{i}' for i in range(1, 15)]
RESPONSE_COLS = OP_COLS + ['extra']
KEY_COLS = ['participant_number', 'List_Number']
LIST_TYPE_MAP = {1: 1, 2: 1, 3: 1, 4: 1, 5: 2, 7: 2, 6: 3, 8: 3}
LIST_TYPE_LABELS = {1: 'mixed', 2: 'pure emotional', 3: 'pure neutral'}
ERROR_CODE_LABELS = {
    -90: 'does_not_exist_or_mix_of_two_images',
    -80: 'another_list',
    -70: 'too_vague_or_unclassified',
    -60: 'unexpected_error',
}


In [ ]:
#| code-summary: "6 — Define file hashing and JSON conversion"

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_jsonable(value):
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value) if not isinstance(value, (str, bytes, dict, list, tuple)) else False:
        return None
    return value


## Load and align the workbook sheets

The final workbook's `rec_itemnos` sheet supplies scored recalled identities. The earlier `recalls` sheet and individual scorer sheets are retained for comparison, not substituted for the adjudicated source.

In [ ]:
#| code-summary: "7 — Read the required workbook sheets"

required_sources = {
    'final_workbook': FINAL_WORKBOOK,
    'similarity_ratings': SIM_EXP1,
    'juliette_scoring': SCORING_JULIETTE,
    'dusan_scoring': SCORING_DUSAN,
    'scoring_discrepancies': SCORING_DISCREPANCIES,
    'filtered_scoring_discrepancies': SCORING_DISCREPANCIES_FILTERED,
}
for label, source_path in required_sources.items():
    assert source_path.exists(), f'Missing {label}: {source_path}'

pres_df = pd.read_excel(FINAL_WORKBOOK, sheet_name='pres_itemnos')
valence_label_df = pd.read_excel(FINAL_WORKBOOK, sheet_name='pres_itemnos_valence')
catlabel_df = pd.read_excel(FINAL_WORKBOOK, sheet_name='pres_catlabels')
recall_df = pd.read_excel(FINAL_WORKBOOK, sheet_name='recalls')
rec_itemnos_df = pd.read_excel(FINAL_WORKBOOK, sheet_name='rec_itemnos')
rec_itemnos_valence_df = pd.read_excel(FINAL_WORKBOOK, sheet_name='rec_itemnos_valence')
rec_catlabels_df = pd.read_excel(FINAL_WORKBOOK, sheet_name='rec_catlabels')
juliette_rec_itemnos_df = pd.read_excel(SCORING_JULIETTE)
dusan_rec_itemnos_df = pd.read_excel(SCORING_DUSAN)
scoring_discrepancies_df = pd.read_excel(SCORING_DISCREPANCIES, sheet_name='Sheet 1')
scoring_actual_disagreements_df = pd.read_excel(
    SCORING_DISCREPANCIES_FILTERED, sheet_name='actual disagreements'
)


In [ ]:
#| code-summary: "8 — Define alignment to presentation rows"

workbook_key_repairs = {}


def align_sheet_keys(df, name):
    if pres_df[KEY_COLS].equals(df[KEY_COLS]):
        return df.copy()

    same_rows = len(df) == len(pres_df)
    same_participants = same_rows and pres_df['participant_number'].equals(
        df['participant_number']
    )
    if same_participants:
        mismatch_mask = (pres_df[KEY_COLS] != df[KEY_COLS]).any(axis=1)
        mismatches = pd.concat(
            [
                pres_df.loc[mismatch_mask, KEY_COLS].add_prefix('pres_'),
                df.loc[mismatch_mask, KEY_COLS].add_prefix(f'{name}_'),
            ],
            axis=1,
        )
        workbook_key_repairs[name] = mismatches
        repaired = df.copy()
        repaired[KEY_COLS] = pres_df[KEY_COLS]
        print(
            f'[INFO] Repaired {name} key labels by row order '
            f'after {len(mismatches)} key mismatch(es).'
        )
        display(mismatches)
        return repaired

    pres_index = pd.MultiIndex.from_frame(pres_df[KEY_COLS])
    sheet_index = pd.MultiIndex.from_frame(df[KEY_COLS])
    missing = list(pres_index.difference(sheet_index))[:10]
    extra = list(sheet_index.difference(pres_index))[:10]
    raise AssertionError(
        f'{name} rows are not aligned and cannot be repaired by row order. '
        f'missing examples={missing}; extra examples={extra}'
    )


In [ ]:
#| code-summary: "9 — Align rows and report the study sample"

valence_label_df = align_sheet_keys(valence_label_df, 'pres_itemnos_valence')
catlabel_df = align_sheet_keys(catlabel_df, 'pres_catlabels')
recall_df = align_sheet_keys(recall_df, 'recalls')
rec_itemnos_df = align_sheet_keys(rec_itemnos_df, 'rec_itemnos')
rec_itemnos_valence_df = align_sheet_keys(rec_itemnos_valence_df, 'rec_itemnos_valence')
rec_catlabels_df = align_sheet_keys(rec_catlabels_df, 'rec_catlabels')
juliette_rec_itemnos_df = align_sheet_keys(juliette_rec_itemnos_df, 'juliette_rec_itemnos')
dusan_rec_itemnos_df = align_sheet_keys(dusan_rec_itemnos_df, 'dusan_rec_itemnos')

print(f'Source lab root: {LAB_ROOT}')
print(f'Trials: {len(pres_df)}')
print(f"Subjects: {pres_df['participant_number'].nunique()}")
print(f"List counts: {pres_df['List_Number'].value_counts().sort_index().to_dict()}")


## Identify stimuli and construct study arrays

The ID map keeps the workbook's global image identities. The original handling of the single missing study event is retained, including its map from recorded slots to compact within-list codes.

In [ ]:
#| code-summary: "10 — Define image labels and category codes"

def normalize_image_label(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text or text.upper() == "MISSING" or text.lower() == "nan":
        return None
    match = re.fullmatch(r"(neg|neu)(\d+)", text.lower())
    if match is None:
        raise ValueError(f"Unexpected image valence label: {value!r}")
    prefix = "NegImage" if match.group(1) == "neg" else "NeuImage"
    return f"{prefix}{int(match.group(2))}.JPG"


def category_from_code(value):
    if pd.isna(value):
        return None
    code = int(float(value))
    if code == 101:
        return "negative"
    if code == 100:
        return "neutral"
    raise ValueError(f"Unexpected category code: {value!r}")


In [ ]:
#| code-summary: "11 — Build and check the image ID map"

image_records = {}
for row_idx in range(len(pres_df)):
    for col in PRES_COLS:
        img_id = pres_df.loc[row_idx, col]
        if pd.isna(img_id) or str(img_id).strip().upper() == "MISSING":
            continue

        img_id = int(img_id)
        filename = normalize_image_label(valence_label_df.loc[row_idx, col])
        category = category_from_code(catlabel_df.loc[row_idx, col])
        assert filename is not None and category is not None
        assert (filename.startswith("Neg") and category == "negative") or (
            filename.startswith("Neu") and category == "neutral"
        )

        previous = image_records.setdefault(img_id, (filename, category))
        assert previous == (filename, category), (
            f"Conflicting labels for image {img_id}: {previous} vs {(filename, category)}"
        )

id_map = pd.DataFrame(
    [
        {"image_id": img_id, "filename": filename, "category": category}
        for img_id, (filename, category) in sorted(image_records.items())
    ]
)
id_map.to_csv(OUTPUT_DIR / "Dupertuys2026_image_id_map.csv", index=False)

NEGATIVE_IDS = set(id_map.loc[id_map["category"] == "negative", "image_id"])
NEUTRAL_IDS = set(id_map.loc[id_map["category"] == "neutral", "image_id"])

assert len(id_map) == 48, f"Expected 48 images, got {len(id_map)}"
assert len(NEGATIVE_IDS) == 24, f"Expected 24 negative images, got {len(NEGATIVE_IDS)}"
assert len(NEUTRAL_IDS) == 24, f"Expected 24 neutral images, got {len(NEUTRAL_IDS)}"
assert id_map.loc[id_map["image_id"] == 39, "filename"].item() == "NeuImage15.JPG"

print(f"Saved image ID map ({len(id_map)} rows)")
print(f"Negative IDs: {sorted(NEGATIVE_IDS)}")
print(f"Neutral IDs: {sorted(NEUTRAL_IDS)}")
id_map.head()


In [ ]:
#| code-summary: "12 — Construct trial metadata and study arrays"

n_trials = len(pres_df)

subject = pres_df["participant_number"].to_numpy(dtype=np.int64).reshape(-1, 1)
list_number = pres_df["List_Number"].to_numpy(dtype=np.int64).reshape(-1, 1)
list_type = np.array(
    [LIST_TYPE_MAP[int(list_no)] for list_no in list_number.flatten()],
    dtype=np.int64,
).reshape(-1, 1)

pres_raw = pres_df[PRES_COLS].replace("MISSING", np.nan)
pres_itemids = pres_raw.fillna(0).to_numpy(dtype=np.int64)
assert pres_itemids.shape == (n_trials, MAX_LIST_LENGTH)

listLength = (pres_itemids > 0).sum(axis=1).reshape(-1, 1).astype(np.int64)

# pres_itemnos are compact serial positions used by jaxcmr. The single MISSING
# presentation gets 0, and later valid items are shifted down by one.
pres_itemnos = np.zeros_like(pres_itemids)
original_to_compact_positions = []
for trial_idx in range(n_trials):
    mapping = {}
    compact_pos = 1
    for original_pos, img_id in enumerate(pres_itemids[trial_idx], start=1):
        if img_id > 0:
            pres_itemnos[trial_idx, original_pos - 1] = compact_pos
            mapping[original_pos] = compact_pos
            compact_pos += 1
    original_to_compact_positions.append(mapping)


In [ ]:
#| code-summary: "13 — Inspect list lengths and array shapes"

short_trials = np.where(listLength.flatten() < MAX_LIST_LENGTH)[0]
if short_trials.size:
    print("Short trials:")
    for idx in short_trials:
        print(
            f"  row={idx}, subject={int(subject[idx, 0])}, "
            f"list={int(list_number[idx, 0])}, length={int(listLength[idx, 0])}"
        )

print(f"subject: {subject.shape}")
print(f"list_number: {list_number.shape}, unique={np.unique(list_number).tolist()}")
print(f"listLength: {listLength.shape}, unique={np.unique(listLength).tolist()}")
print(f"list_type: {list_type.shape}, unique={np.unique(list_type).tolist()}")
print(f"pres_itemnos: {pres_itemnos.shape}")
print(
    f"pres_itemids: {pres_itemids.shape}, "
    f"range=[{pres_itemids[pres_itemids > 0].min()}, {pres_itemids.max()}]"
)


## Score recalls and reconcile sources

The procedure excludes buffers, explicit error codes and non-current-list items, and drops repeated recalls for this experiment. Source comparisons preserve the evidence for choosing the final scored workbook. These are this study's scoring decisions, not universal JAXCMR rules.

In [ ]:
#| code-summary: "14 — Parse recalled identities, positions and categories"

def parse_recalled_item(value):
    if pd.isna(value):
        return None, 'empty'
    text = str(value).strip()
    if not text or text.lower() == 'nan':
        return None, 'empty'
    if text.lower() in {'buff', 'biff'}:
        return None, 'buffer'
    try:
        code = int(float(text))
    except ValueError as exc:
        raise ValueError(f'Unexpected rec_itemnos value: {value!r}') from exc
    if code == 0:
        return None, 'empty'
    if code in ERROR_CODE_LABELS:
        return None, ERROR_CODE_LABELS[code]
    if code < 0:
        return None, 'other_error_code'
    return code, 'item'


def parse_serial_position(value):
    if pd.isna(value):
        return None
    try:
        position = int(float(value))
    except (TypeError, ValueError):
        return None
    return position if position > 0 else None


def parse_category_code(value):
    if pd.isna(value):
        return None
    try:
        code = int(float(value))
    except (TypeError, ValueError):
        return None
    return code if code in {100, 101} else None


In [ ]:
#| code-summary: "15 — Normalize scorer values and map trial items"

def normalize_scoring_value(value):
    if pd.isna(value) or value == '':
        return '<NA>'
    text = str(value).strip()
    try:
        numeric = float(text)
        if numeric.is_integer():
            return str(int(numeric))
    except ValueError:
        pass
    return text


def trial_item_maps(trial_idx):
    item_to_compact = {
        int(img_id): int(pos)
        for pos, img_id in zip(pres_itemnos[trial_idx], pres_itemids[trial_idx])
        if pos > 0 and img_id > 0
    }
    compact_to_itemid = {pos: item for item, pos in item_to_compact.items()}
    item_to_category = {
        int(pres_df.loc[trial_idx, col]): int(catlabel_df.loc[trial_idx, col])
        for col in PRES_COLS
        if not pd.isna(pres_df.loc[trial_idx, col])
        and str(pres_df.loc[trial_idx, col]).strip().upper() != 'MISSING'
    }
    return item_to_compact, compact_to_itemid, item_to_category


In [ ]:
#| code-summary: "16 — Define the final scored-item conversion"

def build_sequences_from_itemnos(
    itemnos_df,
    *,
    response_cols=RESPONSE_COLS,
    drop_repeats=True,
    category_audit=False,
):
    all_trial_recalls = []
    all_trial_itemids = []
    counts = Counter()
    category_mismatches = []

    for trial_idx in range(n_trials):
        subject_id = int(subject[trial_idx, 0])
        list_no = int(list_number[trial_idx, 0])
        item_to_compact, _, item_to_category = trial_item_maps(trial_idx)
        seen_compact = set()
        trial_recalls = []
        trial_itemids = []

        for col in response_cols:
            if col not in itemnos_df.columns:
                continue
            item_id, reason = parse_recalled_item(itemnos_df.loc[trial_idx, col])
            if item_id is None:
                if reason != 'empty':
                    counts[f'ignored_{reason}'] += 1
                continue

            counts['numeric_rec_itemnos'] += 1
            if item_id not in item_to_compact:
                counts['ignored_non_current_list_or_unknown_item'] += 1
                continue

            compact_pos = item_to_compact[item_id]
            if category_audit:
                canonical_category = item_to_category[item_id]
                rec_catlabel_code = parse_category_code(rec_catlabels_df.loc[trial_idx, col])
                if rec_catlabel_code is not None and rec_catlabel_code != canonical_category:
                    category_mismatches.append(
                        {
                            'trial_index': trial_idx,
                            'subject': subject_id,
                            'list_number': list_no,
                            'output_position': col,
                            'item_id': item_id,
                            'canonical_category_code': canonical_category,
                            'rec_itemnos_valence': rec_itemnos_valence_df.loc[trial_idx, col],
                            'rec_catlabel_code': rec_catlabel_code,
                        }
                    )

            if drop_repeats and compact_pos in seen_compact:
                counts['dropped_repeated_items'] += 1
                continue

            seen_compact.add(compact_pos)
            trial_recalls.append(compact_pos)
            trial_itemids.append(item_id)

        all_trial_recalls.append(trial_recalls)
        all_trial_itemids.append(trial_itemids)

    return all_trial_recalls, all_trial_itemids, counts, category_mismatches


In [ ]:
#| code-summary: "17 — Define the earlier serial-position conversion"

def build_sequences_from_serial_recalls(recall_source_df, *, drop_repeats=False):
    all_trial_recalls = []
    all_trial_itemids = []
    counts = Counter()

    for trial_idx in range(n_trials):
        _, compact_to_itemid, _ = trial_item_maps(trial_idx)
        seen_compact = set()
        trial_recalls = []
        trial_itemids = []

        for col in OP_COLS:
            compact_pos = parse_serial_position(recall_source_df.loc[trial_idx, col])
            if compact_pos is None:
                continue
            original_pos = compact_pos
            compact_pos = original_to_compact_positions[trial_idx].get(original_pos)
            if compact_pos is None or compact_pos not in compact_to_itemid:
                counts['serial_positions_out_of_range'] += 1
                continue
            if compact_pos in seen_compact:
                counts['serial_repeated_positions'] += 1
                if drop_repeats:
                    continue
            seen_compact.add(compact_pos)
            trial_recalls.append(compact_pos)
            trial_itemids.append(compact_to_itemid[compact_pos])

        all_trial_recalls.append(trial_recalls)
        all_trial_itemids.append(trial_itemids)

    return all_trial_recalls, all_trial_itemids, counts


In [ ]:
#| code-summary: "18 — Define sequence and scoring comparisons"

def compare_trial_sequences(left_itemids, right_itemids, right_label):
    sequence_diffs = []
    count_diffs = []
    for trial_idx, (left, right) in enumerate(zip(left_itemids, right_itemids)):
        if left != right:
            diff = {
                'trial_index': trial_idx,
                'subject': int(subject[trial_idx, 0]),
                'list_number': int(list_number[trial_idx, 0]),
                'recalls_sheet_itemids': left,
                f'{right_label}_itemids': right,
            }
            sequence_diffs.append(diff)
            if len(left) != len(right):
                count_diffs.append(diff)
    return sequence_diffs, count_diffs


def compare_scoring_cells(left_df, right_df, response_cols=RESPONSE_COLS):
    mismatches = []
    for trial_idx in range(n_trials):
        for col in response_cols:
            if col not in left_df.columns or col not in right_df.columns:
                continue
            left = normalize_scoring_value(left_df.loc[trial_idx, col])
            right = normalize_scoring_value(right_df.loc[trial_idx, col])
            if left != right:
                mismatches.append((trial_idx, col, left, right))
    return mismatches


def nonempty_discrepancy_cells(df):
    cells = set()
    for _, row in df.iterrows():
        if pd.isna(row.get('participant_number')) or pd.isna(row.get('List_Number')):
            continue
        key = (int(row['participant_number']), int(row['List_Number']))
        for col in RESPONSE_COLS:
            if col in df.columns and normalize_scoring_value(row[col]) != '<NA>':
                cells.add((*key, col))
    return cells


In [ ]:
#| code-summary: "19 — Convert final recalls and compare alternative sources"

# Final HDF5 arrays use the adjudicated final-workbook item-ID source.
all_recalls, all_rec_itemids, audit_counts, rec_catlabel_mismatches = (
    build_sequences_from_itemnos(
        rec_itemnos_df,
        response_cols=RESPONSE_COLS,
        drop_repeats=True,
        category_audit=True,
    )
)

# Provenance checks are deliberately separate from HDF5 preparation.
serial_recalls, serial_itemids, serial_counts = build_sequences_from_serial_recalls(
    recall_df, drop_repeats=False
)
final_op_recalls, final_op_itemids, _, _ = build_sequences_from_itemnos(
    rec_itemnos_df, response_cols=OP_COLS, drop_repeats=False
)
juliette_op_recalls, juliette_op_itemids, _, _ = build_sequences_from_itemnos(
    juliette_rec_itemnos_df, response_cols=OP_COLS, drop_repeats=False
)
dusan_op_recalls, dusan_op_itemids, _, _ = build_sequences_from_itemnos(
    dusan_rec_itemnos_df, response_cols=OP_COLS, drop_repeats=False
)

serial_vs_final_diffs, serial_vs_final_count_diffs = compare_trial_sequences(
    serial_itemids, final_op_itemids, 'final_rec_itemnos'
)
serial_vs_juliette_diffs, serial_vs_juliette_count_diffs = compare_trial_sequences(
    serial_itemids, juliette_op_itemids, 'juliette_rec_itemnos'
)
serial_vs_dusan_diffs, serial_vs_dusan_count_diffs = compare_trial_sequences(
    serial_itemids, dusan_op_itemids, 'dusan_rec_itemnos'
)


In [ ]:
#| code-summary: "20 — Summarize scorer disagreements"

final_vs_juliette_cells = compare_scoring_cells(rec_itemnos_df, juliette_rec_itemnos_df)
final_vs_dusan_cells = compare_scoring_cells(rec_itemnos_df, dusan_rec_itemnos_df)
final_vs_juliette_cell_keys = {
    (
        int(subject[trial_idx, 0]),
        int(list_number[trial_idx, 0]),
        col,
    )
    for trial_idx, col, _, _ in final_vs_juliette_cells
}
discrepancy_cells = nonempty_discrepancy_cells(scoring_discrepancies_df)
actual_disagreement_cells = nonempty_discrepancy_cells(scoring_actual_disagreements_df)

final_vs_juliette_choice_counts = Counter()
for trial_idx, col, final_value, juliette_value in final_vs_juliette_cells:
    dusan_value = normalize_scoring_value(dusan_rec_itemnos_df.loc[trial_idx, col])
    if final_value == dusan_value:
        final_vs_juliette_choice_counts['matches_dusan'] += 1
    else:
        final_vs_juliette_choice_counts['matches_neither_scorer'] += 1


In [ ]:
#| code-summary: "21 — Build padded recall arrays and source counts"

max_recalls = max(len(r) for r in all_recalls)
assert max_recalls <= MAX_LIST_LENGTH

recalls = np.zeros((n_trials, MAX_LIST_LENGTH), dtype=np.int64)
rec_itemids = np.zeros((n_trials, MAX_LIST_LENGTH), dtype=np.int64)
for trial_idx, (trial_recalls, trial_rec_itemids) in enumerate(
    zip(all_recalls, all_rec_itemids)
):
    n = len(trial_recalls)
    recalls[trial_idx, :n] = trial_recalls
    rec_itemids[trial_idx, :n] = trial_rec_itemids

rec_catlabel_mismatch_df = pd.DataFrame(rec_catlabel_mismatches)
source_provenance_audit = {
    'recalls_sheet_matches_juliette_scoring_after_itemid_mapping': len(serial_vs_juliette_diffs) == 0,
    'recalls_vs_juliette_sequence_diff_trials': len(serial_vs_juliette_diffs),
    'recalls_vs_juliette_count_diff_trials': len(serial_vs_juliette_count_diffs),
    'recalls_vs_final_rec_itemnos_sequence_diff_trials': len(serial_vs_final_diffs),
    'recalls_vs_final_rec_itemnos_count_diff_trials': len(serial_vs_final_count_diffs),
    'recalls_vs_dusan_sequence_diff_trials': len(serial_vs_dusan_diffs),
    'recalls_vs_dusan_count_diff_trials': len(serial_vs_dusan_count_diffs),
    'final_rec_itemnos_vs_juliette_mismatch_cells': len(final_vs_juliette_cells),
    'final_rec_itemnos_vs_juliette_mismatch_trials': len({trial_idx for trial_idx, *_ in final_vs_juliette_cells}),
    'final_rec_itemnos_vs_dusan_mismatch_cells': len(final_vs_dusan_cells),
    'final_rec_itemnos_vs_dusan_mismatch_trials': len({trial_idx for trial_idx, *_ in final_vs_dusan_cells}),
    'final_rec_itemnos_diff_from_juliette_cells_in_discrepancy_sheet': len(final_vs_juliette_cell_keys & discrepancy_cells),
    'final_rec_itemnos_diff_from_juliette_cells_in_actual_disagreements_sheet': len(final_vs_juliette_cell_keys & actual_disagreement_cells),
    'final_rec_itemnos_diff_from_juliette_choice_counts': dict(final_vs_juliette_choice_counts),
}


In [ ]:
#| code-summary: "22 — Record scoring provenance and known limitations"

recall_source_audit = {
    'canonical_recall_source': 'final_workbook.rec_itemnos',
    'canonical_category_source': 'final_workbook.pres_itemnos + final_workbook.pres_catlabels',
    'counts': dict(audit_counts),
    'serial_sheet_counts': dict(serial_counts),
    'serial_sequence_diff_trials': len(serial_vs_final_diffs),
    'serial_count_diff_trials': len(serial_vs_final_count_diffs),
    'rec_catlabel_mismatch_outputs': len(rec_catlabel_mismatches),
    'source_provenance': source_provenance_audit,
}
prep_gap_assessment = [
    {
        'severity': 'must fix before rerunning models',
        'gap': 'Prepared HDF5 must be regenerated from this audited notebook and carry provenance metadata.',
        'status': 'addressed by this notebook when it executes successfully',
    },
    {
        'severity': 'should fix for reproducibility',
        'gap': 'Some final rec_itemnos cells differ from Juliette and match neither original scorer; the final workbook appears adjudicated, but complete human adjudication notes should be preserved with the raw source data.',
        'status': 'documented as a provenance limitation; not inferable from HDF5 alone',
    },
    {
        'severity': 'acceptable/documented',
        'gap': 'Participant 82 has a row-label mismatch in recall item sheets; row order and all other sheets identify the row as list 4.',
        'status': 'repaired by row-order alignment and recorded in workbook_key_repairs',
    },
    {
        'severity': 'acceptable/documented',
        'gap': 'rec_itemnos_valence/rec_catlabels mislabel item 63 as emotional for a subset of recalls.',
        'status': 'not used as canonical category source; category is derived from pres_itemnos/pres_catlabels',
    },
]


In [ ]:
#| code-summary: "23 — Display aggregate scoring checks"

print('Canonical recall source: final workbook rec_itemnos')
print('Canonical category source: pres_itemnos + pres_catlabels')
print(f'recalls: {recalls.shape}, range=[{recalls.min()}, {recalls.max()}]')
print(f'rec_itemids: {rec_itemids.shape}, range=[{rec_itemids.min()}, {rec_itemids.max()}]')
print('Recall-source audit counts:')
for key, value in sorted(recall_source_audit['counts'].items()):
    print(f'  {key}: {value}')
print('Source-provenance audit:')
for key, value in source_provenance_audit.items():
    print(f'  {key}: {value}')
print(f'Valid studied outputs with rec_catlabels category mismatch: {len(rec_catlabel_mismatches)}')
if not rec_catlabel_mismatch_df.empty:
    display(
        rec_catlabel_mismatch_df.groupby(
            ['item_id', 'canonical_category_code', 'rec_catlabel_code']
        )
        .size()
        .reset_index(name='n')
    )


## Add emotion ratings and check the result

Categories come from the presentation sheets. Continuous ratings are aligned by image identity. Padding is excluded when interpreting a zero category code as neutral.

In [ ]:
#| code-summary: "24 — Load ratings and match them to image identities"

valence_ratings = pd.read_csv(SIM_EXP1 / "valence.csv")
arousal_ratings = pd.read_csv(SIM_EXP1 / "arousal.csv")

valence_data = valence_ratings.drop(
    columns=[valence_ratings.columns[0], "PrivateID"]
).apply(pd.to_numeric, errors="coerce")
arousal_data = arousal_ratings.drop(
    columns=[arousal_ratings.columns[0], "PrivateID"]
).apply(pd.to_numeric, errors="coerce")

img_mean_valence = valence_data.mean(axis=0)
img_mean_arousal = arousal_data.mean(axis=0)

missing_rating_cols = [
    filename
    for filename in id_map["filename"]
    if filename not in img_mean_valence.index or filename not in img_mean_arousal.index
]
assert not missing_rating_cols, f"Missing rating columns: {missing_rating_cols}"

id_to_mean_valence = {
    int(row.image_id): float(img_mean_valence[row.filename])
    for row in id_map.itertuples(index=False)
}
id_to_mean_arousal = {
    int(row.image_id): float(img_mean_arousal[row.filename])
    for row in id_map.itertuples(index=False)
}


In [ ]:
#| code-summary: "25 — Align emotion fields with study events"

valence = np.zeros_like(pres_itemids)
mean_valence = np.zeros((n_trials, MAX_LIST_LENGTH), dtype=np.float64)
mean_arousal = np.zeros((n_trials, MAX_LIST_LENGTH), dtype=np.float64)

for trial_idx in range(n_trials):
    for pos_idx, img_id in enumerate(pres_itemids[trial_idx]):
        img_id = int(img_id)
        if img_id <= 0:
            continue
        if img_id in NEGATIVE_IDS:
            valence[trial_idx, pos_idx] = -1
        mean_valence[trial_idx, pos_idx] = id_to_mean_valence[img_id]
        mean_arousal[trial_idx, pos_idx] = id_to_mean_arousal[img_id]

print(f"valence: {valence.shape}, values={np.unique(valence).tolist()}")
print(
    f"mean_valence: {mean_valence.shape}, "
    f"range=[{mean_valence[mean_valence > 0].min():.2f}, {mean_valence.max():.2f}]"
)
print(
    f"mean_arousal: {mean_arousal.shape}, "
    f"range=[{mean_arousal[mean_arousal > 0].min():.2f}, {mean_arousal.max():.2f}]"
)

neg_mask = valence == -1
neu_mask = (valence == 0) & (pres_itemids > 0)
print(
    f"Negative items: mean_val={mean_valence[neg_mask].mean():.2f}, "
    f"mean_aro={mean_arousal[neg_mask].mean():.2f}"
)
print(
    f"Neutral items:  mean_val={mean_valence[neu_mask].mean():.2f}, "
    f"mean_aro={mean_arousal[neu_mask].mean():.2f}"
)


In [ ]:
#| code-summary: "26 — Check field shapes, codes and padding"

result = {
    "subject": subject,
    "list_number": list_number,
    "listLength": listLength,
    "list_type": list_type,
    "pres_itemnos": pres_itemnos,
    "pres_itemids": pres_itemids,
    "recalls": recalls,
    "rec_itemids": rec_itemids,
    "valence": valence,
    "mean_valence": mean_valence,
    "mean_arousal": mean_arousal,
}

for key, value in result.items():
    assert value.ndim == 2, f"{key} must be 2D. Got shape={value.shape}."
print("[PASS] All entries are 2D")

for key in ["subject", "list_number", "listLength", "list_type"]:
    assert result[key].shape == (n_trials, 1), f"{key} has shape={result[key].shape}"
print("[PASS] Single-column fields have correct shape")

non_negative_fields = {
    "subject",
    "list_number",
    "listLength",
    "list_type",
    "pres_itemnos",
    "pres_itemids",
    "recalls",
    "rec_itemids",
}
for key in non_negative_fields:
    assert result[key].min() >= 0, f"{key} has negative values"
assert set(np.unique(valence)).issubset({-1, 0})
print("[PASS] No unexpected negative values")

assert ((recalls == 0) == (rec_itemids == 0)).all()
print("[PASS] Zero-padding aligned between recalls and rec_itemids")


In [ ]:
#| code-summary: "27 — Check recalled identities and positions"

for trial_idx in range(n_trials):
    nonzero_positions = np.flatnonzero(recalls[trial_idx] == 0)
    if nonzero_positions.size:
        first_zero = int(nonzero_positions[0])
        assert (recalls[trial_idx, first_zero:] == 0).all(), (
            f"Internal zero in compact recall row {trial_idx}"
        )
    assert recalls[trial_idx].max() <= listLength[trial_idx, 0]
print("[PASS] Recalls are compact and within [0, listLength]")

all_image_ids = NEGATIVE_IDS | NEUTRAL_IDS
nonzero_pres = set(pres_itemids[pres_itemids > 0].flatten())
assert nonzero_pres == all_image_ids, (
    f"Presented image set mismatch: missing={all_image_ids - nonzero_pres}, "
    f"unknown={nonzero_pres - all_image_ids}"
)
print(f"[PASS] All {len(all_image_ids)} image IDs are known and presented")

for trial_idx in range(n_trials):
    compact_to_itemid = {
        int(pos): int(img_id)
        for pos, img_id in zip(pres_itemnos[trial_idx], pres_itemids[trial_idx])
        if pos > 0 and img_id > 0
    }
    for compact_pos, recalled_itemid in zip(recalls[trial_idx], rec_itemids[trial_idx]):
        if recalled_itemid == 0:
            continue
        assert compact_to_itemid[int(compact_pos)] == int(recalled_itemid), (
            f"rec_itemids mismatch at trial {trial_idx}: "
            f"position {compact_pos}, item {recalled_itemid}"
        )
print("[PASS] rec_itemids match valid studied rec_itemnos after exclusions")


In [ ]:
#| code-summary: "28 — Check category and rating alignment"

for trial_idx in range(n_trials):
    for pos_idx, img_id in enumerate(pres_itemids[trial_idx]):
        if img_id <= 0:
            assert valence[trial_idx, pos_idx] == 0
        elif img_id in NEGATIVE_IDS:
            assert valence[trial_idx, pos_idx] == -1
        else:
            assert valence[trial_idx, pos_idx] == 0
print("[PASS] Valence correctly aligned with pres_itemids")

assert id_map.loc[id_map["image_id"] == 63, "category"].item() == "neutral"
item_63_codes = []
for trial_idx in range(n_trials):
    for col in PRES_COLS:
        value = pres_df.loc[trial_idx, col]
        if not pd.isna(value) and str(value).strip().upper() != "MISSING" and int(value) == 63:
            item_63_codes.append(int(catlabel_df.loc[trial_idx, col]))
assert set(item_63_codes) == {100}
print(f"[PASS] Item 63 is neutral in all {len(item_63_codes)} presentations")

if not rec_catlabel_mismatch_df.empty:
    assert set(rec_catlabel_mismatch_df["item_id"]) == {63}
    assert set(rec_catlabel_mismatch_df["canonical_category_code"]) == {100}
    assert set(rec_catlabel_mismatch_df["rec_catlabel_code"]) == {101}
    print(
        "[INFO] rec_catlabels mismatch is isolated to item 63: "
        f"{len(rec_catlabel_mismatch_df)} valid studied outputs label it as emotional"
    )

assert (mean_valence[pres_itemids == 0] == 0).all()
assert (mean_arousal[pres_itemids == 0] == 0).all()
assert (mean_valence[pres_itemids > 0] > 0).all()
assert (mean_arousal[pres_itemids > 0] > 0).all()
print("[PASS] Continuous ratings aligned with pres_itemids")


In [ ]:
#| code-summary: "29 — Check participant counts and experimental conditions"

unique_subjects, trial_counts = np.unique(subject, return_counts=True)
assert len(unique_subjects) == 123
assert set(trial_counts) == {4}, f"Unexpected per-subject trial counts: {set(trial_counts)}"
assert n_trials == 492
print("[PASS] 123 subjects x 4 trials = 492 trials")

lt_counts = {label: int((list_type.flatten() == key).sum()) for key, label in LIST_TYPE_LABELS.items()}
assert lt_counts == {"mixed": 246, "pure emotional": 123, "pure neutral": 123}
print(f"[PASS] List type counts: {lt_counts}")

flat_valence = valence[pres_itemids > 0]
for lists, expected, label in [([5, 7], {-1}, "pure emotional"), ([6, 8], {0}, "pure neutral")]:
    mask = np.isin(list_number.flatten(), lists)
    vals = set(valence[mask][pres_itemids[mask] > 0].flatten())
    assert vals == expected, f"{label} valence mismatch: {vals}"

mixed_mask = np.isin(list_number.flatten(), [1, 2, 3, 4])
for trial_vals in valence[mixed_mask]:
    vals = set(trial_vals[trial_vals != 0])
    assert vals == {-1}, "Every mixed list should contain negative items"
    assert np.any(trial_vals == 0), "Every mixed list should contain neutral items"
print("[PASS] List-number-to-valence design checks")


[PASS] 123 subjects x 4 trials = 492 trials
[PASS] List type counts: {'mixed': 246, 'pure emotional': 123, 'pure neutral': 123}
[PASS] List-number-to-valence design checks


## Save data and provenance

Outputs go to `output_dir`, leaving the source workbooks untouched. The audit includes source hashes and local paths for the researcher; inspect newly executed outputs before sharing a copy.

In [ ]:
#| code-summary: "30 — Write the preparation audit"

output_path = OUTPUT_DIR / 'Dupertuys2026.h5'
audit_path = OUTPUT_DIR / 'Dupertuys2026_prep_audit.json'

prep_audit = {
    'prepared_at_utc': datetime.now(timezone.utc).isoformat(),
    'lab_root': LAB_ROOT,
    'sources': {
        label: {
            'path': source_path,
            'sha256': file_sha256(source_path) if source_path.is_file() else None,
        }
        for label, source_path in required_sources.items()
    },
    'workbook_key_repairs': {
        name: df.to_dict(orient='records') for name, df in workbook_key_repairs.items()
    },
    'recall_source_audit': recall_source_audit,
    'prep_gap_assessment': prep_gap_assessment,
}

with open(audit_path, 'w') as f:
    json.dump(to_jsonable(prep_audit), f, indent=2, sort_keys=True)
    f.write('\n')


In [ ]:
#| code-summary: "31 — Save the dataset with provenance attributes"

save_dict_to_hdf5(result, str(output_path))


with h5py.File(output_path, 'a') as h5:
    metadata_json = json.dumps(to_jsonable(prep_audit), sort_keys=True)
    h5.attrs['prepared_at_utc'] = prep_audit['prepared_at_utc']
    h5.attrs['canonical_recall_source'] = 'final_workbook.rec_itemnos'
    h5.attrs['canonical_category_source'] = 'final_workbook.pres_itemnos + final_workbook.pres_catlabels'
    h5.attrs['prep_audit_json'] = metadata_json
    h5.attrs['prep_audit_path'] = str(audit_path)
    h5['data'].attrs['canonical_recall_source'] = 'final_workbook.rec_itemnos'
    h5['data'].attrs['canonical_category_source'] = 'final_workbook.pres_itemnos + final_workbook.pres_catlabels'

print(f'Saved dataset to {output_path}')
print(f'Saved prep audit to {audit_path}')


In [ ]:
#| code-summary: "32 — Inspect the saved HDF5 structure"

with h5py.File(output_path, 'r') as h5:
    print('Saved file structure:')

    def print_structure(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f'  {name}: shape={obj.shape}, dtype={obj.dtype}')

    h5.visititems(print_structure)
    print('HDF5 provenance attributes:')
    for attr in ['prepared_at_utc', 'canonical_recall_source', 'canonical_category_source', 'prep_audit_path']:
        print(f'  {attr}: {h5.attrs[attr]}')


In [ ]:
#| code-summary: "33 — Reload and inspect the prepared arrays"

ds = load_data(str(output_path))
print('Loaded dataset:')
for key, value in ds.items():
    print(f'  {key}: shape={value.shape}, dtype={value.dtype}')

print()
print('First trial:')
print(f"  subject={ds['subject'][0]}")
print(f"  list_number={ds['list_number'][0]}")
print(f"  list_type={ds['list_type'][0]}")
print(f"  pres_itemids={ds['pres_itemids'][0]}")
print(f"  recalls={ds['recalls'][0]}")
print(f"  rec_itemids={ds['rec_itemids'][0]}")


In [ ]:
#| code-summary: "34 — Calculate a descriptive lag-CRP check"

lag_range = MAX_LIST_LENGTH - 1
crp_plus1 = {}
for key, label in [(1, 'mixed'), (2, 'pure emotional'), (3, 'pure neutral')]:
    mask = ds['list_type'].flatten() == key
    subject_values = apply_by_subject(
        ds,
        mask,
        jit(crp.crp, static_argnames=('size')),
        size=1,
    )
    subject_values = np.asarray(jnp.vstack(subject_values))
    crp_plus1[label] = float(np.nanmean(subject_values[:, lag_range + 1]))

print()
print('Subject-mean lag-CRP at +1:')
for label, value in crp_plus1.items():
    print(f'  {label}: {value:.3f}')

assert all(np.isfinite(value) for value in crp_plus1.values())
print('[PASS] Descriptive lag-CRP +1 values are finite')


Subject-mean lag-CRP at +1:
  mixed: 0.232
  pure emotional: 0.254
  pure neutral: 0.241
[PASS] Descriptive lag-CRP +1 values are finite


## Save the emotion feature table

This separate table contains emotion ratings, not semantic similarities. Its rows follow **sorted image IDs**, which have gaps: use the saved ID map to align it rather than indexing it directly by image ID minus one.

In [ ]:
#| code-summary: "35 — Save image-level emotion features and their row order"

rows = []
for row in id_map.itertuples(index=False):
    img_id = int(row.image_id)
    raw_valence = id_to_mean_valence[img_id]
    raw_arousal = id_to_mean_arousal[img_id]
    is_negative = 1.0 if row.category == "negative" else 0.0
    rows.append([raw_valence, raw_arousal, is_negative])

feature_base = np.asarray(rows, dtype=np.float64)
feat_arr = np.column_stack(
    [
        feature_base[:, 0],
        feature_base[:, 1],
        (feature_base[:, 0] - feature_base[:, 0].mean()) / feature_base[:, 0].std(),
        (feature_base[:, 1] - feature_base[:, 1].mean()) / feature_base[:, 1].std(),
        feature_base[:, 2],
    ]
)

np.save(OUTPUT_DIR / "Dupertuys2026_emotion_features.npy", feat_arr)
print(f"Saved Dupertuys2026_emotion_features.npy: {feat_arr.shape}")
print("Columns: mean_valence, mean_arousal, z_valence, z_arousal, is_negative")
print(f"Row order = sorted image IDs: {id_map['image_id'].tolist()}")
